### Guided Lab: Understanding the RAG Pipeline
### Read → Chunk → Embed → Prompt

Welcome! In this hands-on lab you will build a small **Retrieval-Augmented Generation (RAG)** pipeline from scratch, one block at a time.

Meet our (fictional) study group — **Ashi**, **Suyashi**, and **Sid** — three teammates building a RAG assistant that answers questions about *life* using their own notes. We'll use their notes as our sample documents throughout the lab.

**How this lab works**

The lab is split into **4 blocks**, matching the 4 stages of a RAG pipeline:

| Block | Stage | What you'll learn |
|---|---|---|
| 1 | Document Reading | Loading and cleaning raw text |
| 2 | Chunking | Splitting documents into retrievable pieces |
| 3 | Embedding | Turning text into vectors and searching by meaning |
| 4 | Prompting | Combining retrieved chunks into a final LLM prompt |

Each block has **2–3 hands-on exercises**. Every exercise has:
1. A short explanation
2. A `# TODO` code cell for **you** to complete
3. A **Solution** cell so you can check your work



Let's get started!

## ⚙️ Setup

Run the cell below to import everything we need for the whole lab.

In [ ]:
import os
import re
import math
import textwrap
from dataclasses import dataclass, field

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("All set! Environment ready for the RAG lab.")

---
## 📄 Block 1 — Document Reading

Every RAG system starts the same way: **read raw documents into memory** before anything else can happen. In production this might mean reading PDFs, Word docs, web pages, or database rows — but the core idea is always the same: get clean text into your program.

Below are the (fictional) notes from our three teammates. Run the cell to create them as sample files on disk, just like a real document folder.

In [ ]:
# Create a small "documents" folder with notes from Ashi, Suyashi, and Sid
os.makedirs("lab_documents", exist_ok=True)

documents = {
    "ashi_notes.txt": """
    Ashi loves reading about how technology can improve everyday life.
    Today Ashi learned that a Retrieval-Augmented Generation system, or RAG,
    first reads documents before it can answer any question. Ashi wrote in
    her notebook: "A good RAG pipeline starts with clean data, because life
    is too short to feed messy documents into a model." Ashi is now in charge
    of collecting and cleaning all of the team's raw notes.
    """,

    "suyashi_journal.txt": """
    Suyashi is Ashi's teammate, and she is obsessed with chunking strategies.
    In her journal, Suyashi explained to Sid that splitting a long document
    into small chunks helps a RAG system retrieve exactly the piece of
    information that matters, instead of dragging in an entire noisy file.
    Suyashi believes that thoughtful chunking is the secret to a happy RAG
    life, and she keeps experimenting with different chunk sizes every week.
    """,

    "sid_diary.txt": """
    Sid, the third member of the team, focuses on embeddings. Sid wrote:
    "Once Ashi cleans the documents and Suyashi chunks them, my job is to
    turn every chunk of life advice into a vector, so we can compare meaning
    instead of just matching words." Sid believes embeddings are what give a
    RAG system its sense of life-like understanding, connecting a question to
    the right piece of text even when the wording is different.
    """,

    "team_overview.txt": """
    Ashi, Suyashi, and Sid are building a RAG assistant together. It answers
    questions about life advice by reading their notes, chunking them into
    small pieces, embedding every chunk, and finally combining the best
    matches into one prompt for the language model to generate a grounded
    answer. Ashi handles reading, Suyashi handles chunking, and Sid handles
    embedding, together they cover the full RAG pipeline.
    """,
}

for filename, content in documents.items():
    with open(os.path.join("lab_documents", filename), "w") as f:
        f.write(content.strip())

print("Created", len(documents), "files in ./lab_documents/")
print(os.listdir("lab_documents"))

### 🖐️ Exercise 1.1 — Read a single document

**Task:** Write a function `read_document(path)` that opens a text file and returns its contents as a single string.

Then use it to read `lab_documents/ashi_notes.txt` and print the first 150 characters.

In [ ]:
# TODO: complete this function
def read_document(path):
    # 1. open the file at `path`
    # 2. read its full contents
    # 3. return the text as a string
    pass

# TODO: call read_document() on ashi_notes.txt and print a preview
ashi_text = None
print(ashi_text[:150] if ashi_text else "read_document() is not implemented yet")

#### ✅ Solution 1.1

In [ ]:
def read_document(path):
    with open(path, "r") as f:
        return f.read()

ashi_text = read_document("lab_documents/ashi_notes.txt")
print(ashi_text[:150], "...")

### 🖐️ Exercise 1.2 — Read an entire folder of documents

**Task:** Write a function `read_all_documents(folder)` that loops through every `.txt` file in a folder and returns a dictionary mapping `{filename: text}`.

This mirrors how a real RAG ingestion job "crawls" a folder, a Google Drive, or an S3 bucket full of source documents.

In [ ]:
# TODO: complete this function
def read_all_documents(folder):
    docs = {}
    # 1. loop over the filenames in `folder`
    # 2. only handle files ending in .txt
    # 3. read each file and store it in `docs`
    return docs

all_docs = read_all_documents("lab_documents")
print(f"Loaded {len(all_docs)} documents:" if all_docs else "read_all_documents() is not implemented yet")
for name in all_docs:
    print(" -", name)

#### Solution 1.2

In [ ]:
def read_all_documents(folder):
    docs = {}
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            docs[filename] = read_document(os.path.join(folder, filename))
    return docs

all_docs = read_all_documents("lab_documents")
print(f"Loaded {len(all_docs)} documents:")
for name in all_docs:
    print(" -", name)

### 🖐️ Exercise 1.3 — Clean the text

**Task:** Raw text is messy — extra whitespace, repeated newlines, leading/trailing spaces. Write a function `clean_text(text)` that:
1. Collapses multiple whitespace characters (spaces, tabs, newlines) into a single space
2. Strips leading/trailing whitespace

Then apply it to every document in `all_docs`.

> 💡 Hint: `re.sub(r"\s+", " ", text)` collapses any run of whitespace into one space.

In [ ]:
# TODO: complete this function
def clean_text(text):
    # 1. collapse whitespace using re.sub
    # 2. strip leading/trailing spaces
    pass

# TODO: apply clean_text to every document in all_docs
cleaned_docs = {}

for name, text in cleaned_docs.items():
    print(f"--- {name} ---")
    print(text[:120], "...\n")

#### ✅ Solution 1.3

In [ ]:
def clean_text(text):
    text = re.sub(r"\s+", " ", text)
    return text.strip()

cleaned_docs = {name: clean_text(text) for name, text in all_docs.items()}

for name, text in cleaned_docs.items():
    print(f"--- {name} ---")
    print(text[:120], "...\n")

**🎯 Checkpoint:** you now have a dictionary `cleaned_docs` of clean, single-line text for each of Ashi, Suyashi, and Sid's notes. This is exactly the input a chunker expects.

---
## ✂️ Block 2 — Chunking

Feeding an entire document into the embedding step is a bad idea — long documents mix many topics, and a single embedding for a huge file is too "blurry" to match a specific question. Instead, we split documents into smaller **chunks**.

This is Suyashi's favorite part of the pipeline — let's implement her three go-to chunking strategies.

### 🖐️ Exercise 2.1 — Fixed-size chunking

**Task:** Write `chunk_fixed(text, chunk_size)` that splits `text` into consecutive chunks of exactly `chunk_size` characters (the last chunk may be shorter).

Try it on Suyashi's journal with `chunk_size=120`.

In [ ]:
# TODO: complete this function
def chunk_fixed(text, chunk_size=120):
    chunks = []
    # loop over the text in steps of chunk_size and slice it up
    return chunks

suyashi_text = cleaned_docs["suyashi_journal.txt"]
fixed_chunks = chunk_fixed(suyashi_text, chunk_size=120)

for i, c in enumerate(fixed_chunks):
    print(f"[{i}] ({len(c)} chars) {c}")

#### ✅ Solution 2.1

In [ ]:
def chunk_fixed(text, chunk_size=120):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

suyashi_text = cleaned_docs["suyashi_journal.txt"]
fixed_chunks = chunk_fixed(suyashi_text, chunk_size=120)

for i, c in enumerate(fixed_chunks):
    print(f"[{i}] ({len(c)} chars) {c}")

Notice how fixed-size chunking can slice a sentence — or even a word like "**RAG**" — right in half. That's the main downside of this simple approach.

### 🖐️ Exercise 2.2 — Sentence-based chunking

**Task:** Write `chunk_by_sentences(text, sentences_per_chunk)` that:
1. Splits `text` into sentences (split on `.`, `!`, or `?`)
2. Groups every `sentences_per_chunk` sentences into one chunk

This keeps each chunk grammatically whole, which usually reads and embeds better than fixed-size slicing.

> 💡 Hint: `re.split(r'(?<=[.!?]) +', text)` splits on sentence boundaries while keeping punctuation.

In [ ]:
# TODO: complete this function
def chunk_by_sentences(text, sentences_per_chunk=2):
    sentences = []  # TODO: split `text` into a list of sentences
    chunks = []
    # TODO: group `sentences_per_chunk` sentences together into each chunk
    return chunks

sentence_chunks = chunk_by_sentences(suyashi_text, sentences_per_chunk=2)
for i, c in enumerate(sentence_chunks):
    print(f"[{i}] {c}\n")

#### ✅ Solution 2.2

In [ ]:
def chunk_by_sentences(text, sentences_per_chunk=2):
    sentences = re.split(r'(?<=[.!?]) +', text)
    sentences = [s.strip() for s in sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = " ".join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    return chunks

sentence_chunks = chunk_by_sentences(suyashi_text, sentences_per_chunk=2)
for i, c in enumerate(sentence_chunks):
    print(f"[{i}] {c}\n")

### 🖐️ Exercise 2.3 — Overlapping (sliding window) chunking

**Task:** A weakness of both methods above is that important context can fall right on a chunk boundary and get split across two chunks. The fix: let consecutive chunks **overlap**.

Write `chunk_with_overlap(text, chunk_size, overlap)` that slides a window of `chunk_size` characters across `text`, moving forward by `chunk_size - overlap` each step.

In [ ]:
# TODO: complete this function
def chunk_with_overlap(text, chunk_size=120, overlap=30):
    chunks = []
    step = chunk_size - overlap
    # TODO: slide a window across `text` using `step`, slicing `chunk_size` characters each time
    return chunks

overlap_chunks = chunk_with_overlap(suyashi_text, chunk_size=120, overlap=30)
for i, c in enumerate(overlap_chunks):
    print(f"[{i}] {c}\n")

#### ✅ Solution 2.3

In [ ]:
def chunk_with_overlap(text, chunk_size=120, overlap=30):
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        if chunk:
            chunks.append(chunk)
        if i + chunk_size >= len(text):
            break
    return chunks

overlap_chunks = chunk_with_overlap(suyashi_text, chunk_size=120, overlap=30)
for i, c in enumerate(overlap_chunks):
    print(f"[{i}] {c}\n")

**🎯 Checkpoint:** Compare `fixed_chunks`, `sentence_chunks`, and `overlap_chunks` — same source text, three different retrieval behaviors! Overlap trades a little redundancy for much better context preservation, which is why many production RAG systems use it by default.

Now let's build the **final chunk set** we'll use for the rest of the lab — sentence-based chunks from *all* of Ashi, Suyashi, and Sid's documents.

In [ ]:
# Build the master list of chunks across every document, keeping track of the source file
all_chunks = []          # list of chunk text
chunk_sources = []       # which document each chunk came from

for name, text in cleaned_docs.items():
    doc_chunks = chunk_by_sentences(text, sentences_per_chunk=2)
    all_chunks.extend(doc_chunks)
    chunk_sources.extend([name] * len(doc_chunks))

print(f"Total chunks across all documents: {len(all_chunks)}\n")
for i, (c, src) in enumerate(zip(all_chunks, chunk_sources)):
    print(f"[{i}] ({src}) {c}\n")

---
## 🧠 Block 3 — Embedding

Now for Sid's specialty. An **embedding** turns a piece of text into a vector of numbers, positioned so that texts with *similar meaning* end up close together — even if they don't share the exact same words.

We'll use `TfidfVectorizer` from scikit-learn as a simple, fully-offline stand-in for a real embedding model. It's not as powerful as a neural embedding model, but it demonstrates the exact same idea: **text → vector → similarity search**.

### 🖐️ Exercise 3.1 — Turn chunks into vectors

**Task:** Fit a `TfidfVectorizer` on `all_chunks` and transform them into a matrix of vectors. Print the shape of the resulting matrix (`n_chunks x vocabulary_size`).

In [ ]:
# TODO: create a TfidfVectorizer, fit it on all_chunks, and transform all_chunks into vectors
vectorizer = None
chunk_vectors = None

print("Chunk vector matrix shape:", chunk_vectors.shape if chunk_vectors is not None else "not implemented yet")

#### ✅ Solution 3.1

In [ ]:
vectorizer = TfidfVectorizer()
chunk_vectors = vectorizer.fit_transform(all_chunks)

print("Chunk vector matrix shape:", chunk_vectors.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

### 🖐️ Exercise 3.2 — Compare a query to every chunk

**Task:** Write `rank_chunks(query, top_k)` that:
1. Transforms `query` into a vector using the **same** fitted `vectorizer`
2. Computes cosine similarity between the query vector and every chunk vector
3. Returns the `top_k` chunks with the highest similarity, along with their scores

Try it with the query: `"What does Sid think embeddings give a RAG system?"`

In [ ]:
# TODO: complete this function
def rank_chunks(query, top_k=3):
    query_vector = None       # TODO: vectorizer.transform() expects a list, e.g. [query]
    similarities = None       # TODO: cosine_similarity(query_vector, chunk_vectors)
    # TODO: get the indices of the top_k highest similarity scores
    results = []               # list of (chunk_text, source, score)
    return results

query = "What does Sid think embeddings give a RAG system?"
for chunk_text, source, score in rank_chunks(query, top_k=3):
    print(f"({score:.3f}) [{source}] {chunk_text}\n")

#### ✅ Solution 3.2

In [ ]:
def rank_chunks(query, top_k=3):
    query_vector = vectorizer.transform([query])
    similarities = cosine_similarity(query_vector, chunk_vectors)[0]
    top_indices = np.argsort(similarities)[::-1][:top_k]
    results = [(all_chunks[i], chunk_sources[i], similarities[i]) for i in top_indices]
    return results

query = "What does Sid think embeddings give a RAG system?"
for chunk_text, source, score in rank_chunks(query, top_k=3):
    print(f"({score:.3f}) [{source}] {chunk_text}\n")

Notice the top result comes from `sid_diary.txt` — the query matched on **meaning-bearing words** ("Sid", "embeddings", "RAG"), even though the sentence is phrased differently from the original text.

### 🖐️ Exercise 3.3 — Build a mini vector store

**Task:** Real RAG systems store embeddings in a **vector database** (e.g. Pinecone, Weaviate, FAISS, pgvector) so they can be searched quickly at scale. Build a simple `VectorStore` class with:
- `add(text, source, vector)` — store one chunk
- `search(query, vectorizer, top_k)` — return the top-k most similar chunks to `query`

This is the same idea as `rank_chunks` above, just packaged as a reusable object — the way it would look in a real application.

In [ ]:
# TODO: complete this class
class VectorStore:
    def __init__(self):
        self.texts = []
        self.sources = []
        self.vectors = []   # list of vectors (rows)

    def add(self, text, source, vector):
        # TODO: append text/source/vector to the store's lists
        pass

    def search(self, query, vectorizer, top_k=3):
        # TODO: transform the query, stack self.vectors into a matrix,
        # compute cosine similarity, and return the top_k (text, source, score) tuples
        pass

# Build and populate the store
store = VectorStore()
# TODO: loop over all_chunks / chunk_sources / chunk_vectors and call store.add(...) for each

results = store.search("What is Suyashi's secret to a happy RAG life?", vectorizer, top_k=2)
for r in (results or []):
    print(r)

#### ✅ Solution 3.3

In [ ]:
class VectorStore:
    def __init__(self):
        self.texts = []
        self.sources = []
        self.vectors = []

    def add(self, text, source, vector):
        self.texts.append(text)
        self.sources.append(source)
        self.vectors.append(vector)

    def search(self, query, vectorizer, top_k=3):
        query_vector = vectorizer.transform([query])
        matrix = np.vstack(self.vectors)
        similarities = cosine_similarity(query_vector, matrix)[0]
        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [(self.texts[i], self.sources[i], similarities[i]) for i in top_indices]

store = VectorStore()
dense_vectors = chunk_vectors.toarray()
for text, source, vector in zip(all_chunks, chunk_sources, dense_vectors):
    store.add(text, source, vector)

results = store.search("What is Suyashi's secret to a happy RAG life?", vectorizer, top_k=2)
for text, source, score in results:
    print(f"({score:.3f}) [{source}] {text}\n")

**🎯 Checkpoint:** you now have a working (if simplified) retrieval engine: documents → chunks → vectors → similarity search. This `store.search()` call is exactly what happens inside a production RAG system's retrieval step — just with a more powerful embedding model and a real vector database underneath.

---
## ✍️ Block 4 — Building the Final Prompt

This is where **retrieval meets generation**. Given a user's question, we:
1. **Retrieve** the most relevant chunks from the vector store
2. **Augment** the question by inserting those chunks as context
3. **Send** the combined prompt to a language model to generate a grounded answer

Let's build this step by step.

### 🖐️ Exercise 4.1 — Write a `retrieve()` function

**Task:** Write `retrieve(query, top_k)` that wraps `store.search()` and returns just the list of chunk texts (no scores/sources needed for this step).

In [ ]:
# TODO: complete this function
def retrieve(query, top_k=3):
    results = None   # TODO: call store.search(...)
    return []          # TODO: return just the chunk text from each result

question = "How does the team's RAG assistant give life advice?"
retrieved = retrieve(question, top_k=3)
for r in retrieved:
    print("-", r)

#### ✅ Solution 4.1

In [ ]:
def retrieve(query, top_k=3):
    results = store.search(query, vectorizer, top_k=top_k)
    return [text for text, source, score in results]

question = "How does the team's RAG assistant give life advice?"
retrieved = retrieve(question, top_k=3)
for r in retrieved:
    print("-", r)

### 🖐️ Exercise 4.2 — Build the final RAG prompt

**Task:** Write `build_prompt(query, context_chunks)` that formats a final prompt string containing:
- A short system instruction telling the model to answer **only** using the provided context
- The retrieved chunks, clearly separated
- The user's original question

This is the exact text that would be sent to an LLM API (like Claude or GPT) as the final "augmented" prompt.

In [ ]:
# TODO: complete this function
def build_prompt(query, context_chunks):
    context_block = ""  # TODO: join context_chunks into a numbered or bulleted block of text
    prompt = ""          # TODO: combine instructions + context_block + query into one string
    return prompt

final_prompt = build_prompt(question, retrieved)
print(final_prompt)

#### ✅ Solution 4.2

In [ ]:
def build_prompt(query, context_chunks):
    context_block = "\n".join(f"[{i+1}] {chunk}" for i, chunk in enumerate(context_chunks))
    prompt = textwrap.dedent(f"""
    You are a helpful assistant. Answer the question using ONLY the context
    below. If the answer isn't in the context, say you don't know.

    Context:
    {context_block}

    Question: {query}

    Answer:
    """).strip()
    return prompt

final_prompt = build_prompt(question, retrieved)
print(final_prompt)

### 🖐️ Exercise 4.3 — Run the full RAG pipeline end-to-end

**Task:** Write `rag_pipeline(query, top_k)` that chains everything together:
1. `retrieve()` the top chunks
2. `build_prompt()` from those chunks
3. "Generate" an answer — since we don't have a live LLM connected in this lab, use the provided `mock_llm_call()` function below to simulate the final generation step

Then run the full pipeline on **your own question** about Ashi, Suyashi, Sid, or their RAG project!

In [ ]:
def mock_llm_call(prompt):
    """Stands in for a real LLM API call (e.g. Anthropic/OpenAI) so this lab runs offline.
    A real implementation would send `prompt` to a chat completion endpoint and return the reply."""
    context_lines = [line for line in prompt.split("\n") if line.strip().startswith("[")]
    combined = " ".join(line.split("] ", 1)[-1] for line in context_lines)
    return f"(mock LLM answer, grounded in retrieved context)\n{combined[:280]}..."

# TODO: complete this function
def rag_pipeline(query, top_k=3):
    chunks = None    # TODO: call retrieve()
    prompt = None    # TODO: call build_prompt()
    answer = None    # TODO: call mock_llm_call()
    return prompt, answer

my_question = "Ask your own question here, e.g. 'What is Ashi responsible for in the RAG pipeline?'"
prompt, answer = rag_pipeline(my_question, top_k=2)
print("=== FINAL PROMPT SENT TO LLM ===\n")
print(prompt)
print("\n=== GENERATED ANSWER ===\n")
print(answer)

#### ✅ Solution 4.3

In [ ]:
def rag_pipeline(query, top_k=3):
    chunks = retrieve(query, top_k=top_k)
    prompt = build_prompt(query, chunks)
    answer = mock_llm_call(prompt)
    return prompt, answer

my_question = "What is Ashi responsible for in the RAG pipeline?"
prompt, answer = rag_pipeline(my_question, top_k=2)
print("=== FINAL PROMPT SENT TO LLM ===\n")
print(prompt)
print("\n=== GENERATED ANSWER ===\n")
print(answer)

**🎯 Checkpoint — you just built a complete, working RAG pipeline!**

```
Documents (Ashi, Suyashi, Sid's notes)
        │  Block 1: read + clean
        ▼
   Chunks of text
        │  Block 2: fixed / sentence / overlap chunking
        ▼
   Chunk vectors in a vector store
        │  Block 3: embed + similarity search
        ▼
   Retrieved chunks  +  user question
        │  Block 4: build_prompt()
        ▼
   Final augmented prompt  →  LLM  →  Grounded answer
```

### 🚀 Try it yourself

Go back to Exercise 4.3 and run `rag_pipeline()` with a few of your own questions, for example:
- `"What does Suyashi think is the secret to a happy RAG life?"`
- `"Why does Sid say embeddings matter for a RAG system?"`
- `"Who is responsible for chunking on the team?"`

Watch how the **retrieved chunks change** depending on the wording of your question — that's the retrieval step doing its job.

### 🔁 From toy lab to production

| In this lab | In production, swap in |
|---|---|
| `TfidfVectorizer` | A neural embedding model (e.g. OpenAI `text-embedding-3`, Sentence-Transformers, Voyage AI) |
| Python list `VectorStore` | A real vector database (FAISS, Pinecone, Weaviate, Chroma, pgvector) |
| `mock_llm_call()` | A real LLM API call (e.g. the Anthropic Messages API) with the built prompt |
| Sentence-based chunking | Often combined with overlap, plus document-structure-aware chunking (headers, tables) |

Great work — you've now implemented every core stage of a RAG pipeline: **reading, chunking, embedding, and prompting**, just like Ashi, Suyashi, and Sid did for their own project. Thank you for completing the lab! 🎉